# Comb explorer — in a notebook

`plots.comb_widget.comb_explorer(frame, t0=..., dur=...)` builds the interactive
comb-explorer page for any `tdseries` Frame and displays it in the cell.

It discovers the audio entry and **every** rotor-speed track by itself, and ships
all of them into one page:

* the **carrier** dropdown picks which track drives the combs and the strip
  demodulation (`motors_measured` vs `motors_command` vs a refined track, by eye,
  without a rebuild);
* the **microphone channel** dropdown is in-page state (the CLI has to split
  channels across sibling files because a written page has a 9 MB budget; a
  notebook has no budget, only the size of the saved output).

The page is self-contained: no external host, no fetch, no CDN. It renders inside
a `srcdoc` iframe, which is what makes JupyterLab execute it at all and what keeps
two widgets in two cells from colliding.

**Clear the outputs before committing this notebook** — the payload is tens of
megabytes and it is saved into the `.ipynb`.

In [ ]:
from data_processing import sources
from plots.comb_widget import comb_explorer, discover

## 1. DREGON — free flight, room 1

The published frames stream from R2 (`dload`); nothing is read from a raw tree.

In [ ]:
dregon = next(
    f
    for f in sources.iter_recording_frames("DREGON", splits=["in_flight_noise"])
    if f["meta"]["recording_id"] == "free-flight_nosource_room1"
)
print(discover(dregon).describe())

`discover` finds three carriers here: the measured tachometer, the cleaned
flight-controller command, and the raw command. Switch between them in the page
and watch the comb move against the ridge.

In [ ]:
comb_explorer(dregon, t0=22.56481, dur=8.0, ks="1-60")

## 2. Michael's DJI Matrice 100 — FLY124

The same call on a different rig: 4 rotors, 8 microphones, one calibrated `rps`
track. Everything else — rotor count, microphone count, sample rate — comes from
the Frame.

In [ ]:
fly124 = next(
    f
    for f in sources.iter_recording_frames("michaels")
    if f["meta"]["recording_id"] == "FLY124"
)
print(discover(fly124).describe())

In [ ]:
comb_explorer(fly124, t0=52.0, dur=8.0, ks="1-60")

## 3. Knobs

| argument | default | what it does |
|---|---|---|
| `t0`, `dur` | `0.0`, whole audio | window, seconds from the start of the frame's audio (`absolute=True` for absolute times) |
| `channels` | `"auto"` | `"auto"` = `avg` + the loudest single mic; also `"all"`, `"avg"`, `"0,3,avg"`, `[0, 3]` |
| `rps_keys` | `None` | restrict / reorder the carriers, e.g. `["motors_measured", "motors_command"]`; the first one sets the k ceiling |
| `ks`, `k_max` | `"1-100"`, `100` | harmonics; keep the set contiguous — a hole in k is the one failure this page must not have |
| `segs` | `(0.1, 0.5, 2.0)` | strip segment lengths, seconds |
| `decim` | `32` | envelope decimation; 32 = 1378 Hz = ±7.25 rev/s of usable offset even at k = 95 |
| `ylim` | `6.0` | strip half-range, rev/s |
| `gap_tol` | `0.5` | a telemetry hole longer than this inside the window is an error, not a straight line |
| `nfft`, `fmax` | `2048`, `10000` | spectrogram |
| `jobs` | 4 | demodulation workers |
| `height` | `1500` | iframe height in pixels (it also grows to its content and is drag-resizable) |
| `warn_mb` | `30.0` | warn when the cell output gets this big |

To slice a Frame down first — one microphone, two rotors — use the `tdseries`
dim slicer and pass the result straight in:

```python
comb_explorer(dregon.slice["mic", 0:1], t0=22.56481, dur=8.0)
comb_explorer(dregon.slice["rotor", 0:2], t0=22.56481, dur=8.0)
```

The file-writing CLI for the same page is
`python scripts/displacement/comb_explorer.py --recording ... --t0 ... --dur ...`.